# Train YOLO on the Chess Pieces Dataset

Notebook-only workflow for Colab, Kaggle, or Jupyter.

Dataset reference:
- https://platform.ultralytics.com/sai-satish/datasets/chess-pieces

This notebook does not include the Streamlit app. It only prepares the dataset, trains YOLO, evaluates it, and runs a sample prediction.

In [ ]:
%pip install -q ultralytics requests Pillow

Make sure this notebook is running from the repository root, or change `REPO_ROOT` below to point at this project.

In [ ]:
from pathlib import Path
import os

REPO_ROOT = Path.cwd()
os.chdir(REPO_ROOT)
print('Working directory:', REPO_ROOT.resolve())
print('NDJSON exists:', (REPO_ROOT / 'chess-pieces.ndjson').exists())

In [ ]:
assert (Path('chess-pieces.ndjson')).exists(), 'Put chess-pieces.ndjson in the repo root before running training.'

Convert the Ultralytics NDJSON export into a normal local YOLO dataset layout.

In [ ]:
!python scripts/prepare_ultralytics_ndjson.py --input chess-pieces.ndjson --output data/chess-pieces

Train the detector. `yolo11n.pt` is the compatibility default. If your installed Ultralytics version supports it and you want the newest family, change this to `yolo26n.pt`.

In [ ]:
!python train_yolo.py --data data/chess-pieces/data.yaml --model yolo11n.pt --epochs 40 --imgsz 640 --batch 16

Run validation once training finishes.

In [ ]:
from ultralytics import YOLO

best_weights = Path('runs/detect/chess-pieces/weights/best.pt')
print('Best weights:', best_weights.resolve())
model = YOLO(str(best_weights))
metrics = model.val(data='data/chess-pieces/data.yaml')
metrics

Run a sample prediction on one validation image.

In [ ]:
from IPython.display import display
from PIL import Image

sample_images = sorted((Path('data/chess-pieces/images/val')).glob('*'))
sample_path = sample_images[0]
print('Sample image:', sample_path)
prediction = model.predict(source=str(sample_path), conf=0.25, save=True, verbose=False)
display(Image.open(sample_path))
prediction